In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [2]:
df=pd.read_csv("../data/BankNote_Authentication.csv")
                 

In [3]:
print("Shape:", df.shape)

Shape: (1372, 5)


In [4]:
print(df["class"].value_counts())

class
0    762
1    610
Name: count, dtype: int64


In [5]:
X = df.drop(columns="class")
y = df["class"]
features = list(X.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model.fit(X_train, y_train)
print("Model trained")

Model trained


In [6]:
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print("Test accuracy:", round(acc, 4))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

Test accuracy: 0.9709
Confusion matrix:
 [[145   8]
 [  0 122]]
              precision    recall  f1-score   support

           0       1.00      0.95      0.97       153
           1       0.94      1.00      0.97       122

    accuracy                           0.97       275
   macro avg       0.97      0.97      0.97       275
weighted avg       0.97      0.97      0.97       275



In [7]:
DEFAULTS = {'variance': 0.0, 'skewness': 0.0, 'curtosis': 0.0, 'entropy': 0.0}
POS, NEG = "Forged banknote", "Genuine banknote"

In [8]:
import gradio as gr

def predict_gradio(*values):
    row = pd.DataFrame([list(values)], columns=features)
    p = model.predict_proba(row)[0][1]
    return {POS: float(p), NEG: float(1 - p)}

inputs = [gr.Number(label=f, value=DEFAULTS.get(f, 0)) for f in features]

demo = gr.Interface(
    fn=predict_gradio, inputs=inputs, outputs=gr.Label(num_top_classes=2),
    title="Banknote Authenticity Checker",
    description=f"Logistic Regression | test accuracy: {acc*100:.1f}%. "
                "Features come from wavelet-transformed banknote images.")
demo.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [9]:
from flask import Flask, request, render_template_string
from werkzeug.serving import make_server
import threading

try:
    flask_server.shutdown()
except NameError:
    pass

app = Flask(__name__)

PAGE = """
<!doctype html>
<title>Banknote Checker</title>
<style>
  body{font-family:Arial;max-width:520px;margin:30px auto;padding:0 12px}
  label{display:block;margin-top:10px;font-weight:bold}
  input{width:100%;padding:7px;box-sizing:border-box}
  button{margin-top:16px;padding:10px 18px;cursor:pointer}
  .res{margin-top:20px;padding:14px;background:#eef;border-radius:6px}
</style>
<h2>Banknote Authenticity Checker</h2>
<p>Logistic Regression &middot; test accuracy {{ acc }}%</p>
<form method="post">
  {% for f in features %}
    <label>{{ f }}</label>
    <input type="number" step="any" name="{{ f }}" value="{{ vals[f] }}" required>
  {% endfor %}
  <button type="submit">Predict</button>
</form>
{% if result %}
  <div class="res"><b>{{ result }}</b><br>Probability: {{ prob }}%</div>
{% endif %}
"""

@app.route("/", methods=["GET", "POST"])
def index():
    vals = {f: DEFAULTS.get(f, 0) for f in features}
    result = prob = None
    if request.method == "POST":
        vals = {f: float(request.form[f]) for f in features}
        p = model.predict_proba(pd.DataFrame([[vals[f] for f in features]], columns=features))[0][1]
        result = POS if p >= 0.5 else NEG
        prob = round(max(p, 1 - p) * 100, 2)
    return render_template_string(PAGE, features=features, vals=vals,
                                  result=result, prob=prob, acc=round(acc * 100, 1))

flask_server = make_server("127.0.0.1", 5000, app)
threading.Thread(target=flask_server.serve_forever, daemon=True).start()
print("Open http://127.0.0.1:5000 in your browser")

Open http://127.0.0.1:5000 in your browser
